## to do:

QoL
- plot nodes at any level of transform, see all_nodes
- automatically perform max. level transform
- plot single node by path
- time [s] instead of coefficient number
- automatically perform nodes2data on inverse
- fix plots

studentship:
- sensitivity analysis
- testing/tuning thresholding values (classification without machine learning)
- check out tree diagram
- sounds like a report. manuscript should be concise.
- explain the problem you're trying to address in the intro. as well as physics (materials and methods.)
- trying to find a quantitative relationship between cavitation and physical effects.
- write about alternative methods to analyse pcd data
- create filter diagram for wpt in intro, explaining the naming conventions for the levels, node number, coefficient number, etc.
- add entropy and other features that we will use. 
- write up feature selection
- use the silhouette score to test k-means with different numbers of clusters with different types of signals.
- find fundamental papers that I used for feature selection and clustering. (send them)
- generate dynamics data

algorithm
- stick with minimum entropy, find good filter (db order). implement algorithm that does this.
- test with different number of cycles.
- test normalised energy
- use tree graph to make filter bank diagram.

general coding:
- adaptive algorithm

writing:
- write about k-means
- add stuff to the literature review about the other works.

numerical results section:
- stable, stable with weak transients, fully transient (doesn't occur in practice)
- add noise to each
- might be able to do something with the database

latest
- talk about starting points in k-means
- maximum 12 pages
- minimum level of decomposition
- change colours of the flowchart
- change order of wavelet until clustering doesn't improve (find paper)
- write up the results using one stable and one transient signal
- test with pcd data
- conclusion
-  
- plotly?

to do
- define PCA
- save figures as ps or pdf
- make a table specifying which parameters were used to solve the gilmore-akulichev model.
- use (j, k) notation.
- overlay graphs and spectrums
- plot bar charts for the cluster centres.
- k-means 3d feature space plots
- report k-means parameters, look at how sensitive the algorithm is to each

done
- use smaller fontsize for algorithm, change algorithm to the simpler one without the diamond.
- move pcd data intro to start of methods. merge with existing description of features we expect to find in the signal
- change color and size of nodes in tree graph depending on the label of the node
- include fft of the reconstructions to show which features have been selected
- make a figure with nodes and classification
- define what kind of noise was added. additive white gaussian noise to simulate measurement noise. $20\log{\frac{\mu}{\sigma_{noise}}}=10$ signal to noise ratio

SOMs
- don't use numbers e.g. 2 dimensions, 3 clusters.
- cite classical reference for self-organising maps
- explain signals used for feature selection. choose wider frequency range, something that reflects the real life uses of ultrasound. go from 0.5 to 2 MHz.
- talk about variations on the neural network. refer to paper that uses som's to do feature selection.
- convergence analysis on self-organising maps with starting points, number of signals. see which are the best features in each case.
- decide whether to use SOMs

write-up
- don't start (conclusion not summary) with overall
- american english
- use a consistent tense (usually past)
- use commands
- use feature names instead of cluster number
- look at kurtosis to separate measurement noise from broadband

# Decompose a signal to its components at different frequency bands with WPT

In this notebook we aim to test WPT for extracting different components of a signal based on the energy, kurtosis, and crest factor of different frequency bands. The WPT library requires python3.6+ version. The following cell checks this requirement.

In [ ]:
import sys

if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

Import the wavelet_transform package and required libraries.

In [ ]:
sys.path.append("..")
from wavelet_transform.wpt import transform as wpt
from wavelet_transform.utils import synthetic_data as sd
from wavelet_transform.wpt import inverse_transform as inv
from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.statistical_features as stats
from wavelet_transform.utils.extensions import fft as ext_fft

# matplotlib.use('nbagg')
import matplotlib.pyplot as plt
import numpy as np
import pywt
import scipy.stats as sc

Specify the parameters for generating a synthetic signal and to perform wavelet packet transform.

In [ ]:
# keep the same
sampling_rate = 100  # unit: Hz

# PARAMETERS TO CHANGE
amplitudes = [1, 0.8, 0.4]  # amplitude of each sine wave
frequencies = [5, 15, 25]  # frequency content of signal
phases = [0, np.deg2rad(45), np.deg2rad(120)]  # phase of each sine wave

duration = 1000 / min(frequencies)  # duration of wave

trans_amp = [10 for _ in range(5)]  # amplitudes of transients
n_trans = len(trans_amp)  # number of transients
trans_dur = [0.05 for _ in range(n_trans)]  # duration of transients
trans_time = [
    duration * i / (n_trans + 1) for i in range(1, n_trans + 1)
]  # times of transients

std_noise = 0.1

wavelet = "db20"
levels = 5
mode = "symmetric"

sort_by = 'energy'  # 'energy', 'kurtosis', 'crest', 'False' -> freq
sort = "rec"  # 'rec' (reconstructed signal) 'coef' (transform coefficients)
n_highest = 3

fft_lim = 50  # increase value to view more of the fft plots

Generate synthetic multitonal signal with/without additive noise. We will fft the signal to double check the frequency contents.

In [ ]:
# generate sound wave
data, t_plot = sd.sound_wave(amplitudes, frequencies, phases, duration, sampling_rate)
signal_length = len(data)
data += sd.transients(trans_amp, trans_time, trans_dur, duration, sampling_rate)
data += sd.noise(std_noise,duration,sampling_rate)
data *= np.hamming(signal_length)

fft_data, freq = ext_fft(data, sampling_rate)
# plot sound wave and fft
fig, ax = plt.subplots(3, figsize=(11, 12))
ax[0].plot(t_plot, data)
ax[0].set_title("Data")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Amplitude")
ax[0].grid()

ax[1].stem(
    freq,
    np.abs(fft_data),
    linefmt="b",
    markerfmt=" ",
    basefmt="-b",
)
ax[1].set_title("FFT of data - amplitude")
ax[1].set_xlabel("Frequency [Hz]")
ax[1].set_ylabel("Amplitude [DL]")
ax[1].set_xlim(0, sampling_rate / 2)
ax[1].grid()

ax[2].plot(freq, np.angle(fft_data, deg=True))
ax[2].set_title("FFT of data - phase")
ax[2].set_xlabel("Frequency [Hz]")
ax[2].set_ylabel("Phase [rad]")
ax[2].set_xlim(0, sampling_rate / 2)
ax[2].grid()
fig.tight_layout()
plt.show()

### Frequency content of reconstructed subbands
Decompose signal using discrete wavelet packet transform and sort the nodes in order of energy, kurtosis, or crest factor.

functions used for sorting

In [ ]:
def sort_nodes(rec, sort="rec", sort_by="energy"):
    """
    choose how to sort the subbands

    Parameters
    ----------
    rec: tuple
        tuple containing (coefficients,reconstructed_wave,fft_of_rec_wave,energy,kurtosis,crest)
    sort: str
        sort by 'energy', 'kurtosis', or 'crest' factor
    coef_or_rec: str
        sort by calculating attributes of the node coefficients or the reconstructed wave

    returns
    -------
    rec: tuple
        sorted input
    """
    if sort == "rec":
        if sort_by == "energy": 
            return sorted(rec, key=lambda x: x[3], reverse=True)
        elif sort_by == "kurtosis":
            return sorted(rec, key=lambda x: x[4], reverse=True)
        elif sort_by == "crest":
            return sorted(rec, key=lambda x: x[4], reverse=True)
            


    elif sort == "coef":
        if sort_by == "energy":
            return sorted(rec, key=lambda x: x[0].energy, reverse=True)
        if sort_by == "kurtosis":
            return sorted(rec, key=lambda x: x[0].kurtosis, reverse=True)
        if sort_by == "crest":
            return sorted(rec, key=lambda x: x[0].crest_factor, reverse=True)

Transform and sort nodes

In [ ]:
# WPT transform
tr_obj = wpt.dwpt(data, levels, wavelet, mode)
tr_obj.calc_frequencies(sampling_rate)

# reconstruct each node separately
rec = []

for node in tr_obj.get_level(order="freq"):
    inverse = inv.inv_node(node)
    inverse_signal_length = len(inverse)
    fft = ext_fft(inverse)
    energy = stats.energy(inverse)
    kurtosis = sc.kurtosis(inverse)
    crest = stats.crest_factor(inverse)
    entropy = stats.entropy(inverse)
    rec.append((node, inverse, fft, energy, kurtosis, crest, entropy))

plot the **`n_highest`** energy components (subbands).

In [ ]:
def plot_reconstructions(rec, n_highest=None, group=None):
    partial_rec = np.zeros(signal_length)  # track sum of highest energy subbands
    if n_highest is None:
        n_highest = len(rec)
    for i, rec in enumerate(rec[:n_highest]):  # highest energy, kurtosis, etc.
        node, rec_wave, fft, energy, kurtosis, crest, entropy = rec
        partial_rec += rec_wave  # full reconstruction
        fig, ax = plt.subplots(
            2, figsize=(11, 8)
        )  # plot reconstructed wave and its fft
        freq_r = node.frequency_range
        ax[0].plot(t_plot, rec_wave)
        title = f"Reconstructed subband [{freq_r[0]}- {freq_r[1]} Hz]"
        if group is not None:
            title = f"Reconstructed subband [{freq_r[0]}- {freq_r[1]} Hz] group: {group[i]}"
        ax[0].set_title(title)
        ax[0].set_xlabel("time")
        ax[0].set_ylabel("amplitude")
        ax[1].stem(
            freq,
            np.abs(fft),
            linefmt="b",
            markerfmt=" ",
            basefmt="-b",
        )
        ax[1].set_title("FFT of reconstructed subband")
        ax[1].set_xlabel("Frequency")
        ax[1].set_ylabel("Power")
        ax[1].set_xlim(
            max(0, freq_r[0] - fft_lim), min(sampling_rate / 2, freq_r[1] + fft_lim)
        )
        ax[1].grid()
        fig.tight_layout()
        plt.show()

        # print info
        print(f"energy of recovered signal: {energy}")
        print(f"kurtosis of recovered signal: {kurtosis}")
        print(f"crest factor of recovered signal: {crest}")
        print(f"entropy of recovered signal: {entropy}")
        print()
        print(f"energy of node coefficients: {node.metric("energy")}")
        print(f"kurtosis of node coefficients: {node.metric("kurtosis")}")
        print(f"crest factor of node coefficients: {node.metric("crest_factor")}")
        print(f"entropy of node coefficinets: {node.metric("entropy")}")
    return partial_rec

run = False
if run is True:
    # sort reconstructed waves
    sorted_rec = sort_nodes(rec, sort, sort_by)
    print(f"sorted by: {sort_by} of {sort}")
    # plot highest energy subbands
    partial_rec = plot_reconstructions(sorted_rec,n_highest)

### heisenberg visualisation

Making an heisenberg plot of the last level of dwpt

In [ ]:
plot.heisenberg(tr_obj, max(t_plot))

### Clustering

k-means clustering algorithm with energy, kurtosis, and crest factor as the input

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# training data
data_train = []
for node, inverse, _, energy, kurtosis, crest, entropy in rec:
    data_train.append(
        (energy, kurtosis, crest, entropy)
        )

scaler = StandardScaler()
data_train_normalised = scaler.fit_transform(data_train)

# training k-means algorithm
no_clusters = 5
kmeans = KMeans(n_clusters=no_clusters)
kmeans.fit(data_train_normalised)

# get labels and clusters
labels = kmeans.labels_
centers = scaler.inverse_transform(kmeans.cluster_centers_)

### plot result

In [ ]:
# plot regular k-means
plt.bar([node.frequency_range[0] for node,*_ in rec], labels)
plt.title('Group Number of each Node vs Frequency')
plt.xlabel('Lower Frequency of Node [Hz]')
plt.ylabel('Group Number')
plt.show()

print("Cluster labels:", labels)
print(f"highest energy: {np.argmax(centers[:,0])}")
print(f'highest kurtosis: {np.argmax(centers[:,1])}')
print(f'highest crest factor: {np.argmax(centers[:,2])}')

### PCA

In [ ]:
# apply PCA to training data
pca = PCA(n_components=3)
data_pca = pca.fit_transform(data_train_normalised)

# training k-means algorithm on the reduced data
kmeans_pca = KMeans(n_clusters=no_clusters)
kmeans_pca.fit(data_pca)

labels_pca = kmeans_pca.labels_
centers_pca = scaler.inverse_transform(pca.inverse_transform(kmeans_pca.cluster_centers_))

### Plot PCA Result

In [ ]:
# plot pca k-means
lower_freqs = [node.frequency_range[0] for node,*_ in rec]
plt.bar([node.frequency_range[0] for node,*_ in rec], labels_pca)
plt.tick_params
plt.title('Group Number of each Node vs Frequency')
plt.xlabel('Lower Frequency of Node [Hz]')
plt.ylabel('Group Number')
plt.show()

eigenvectors = pca.components_
eigenvalues = pca.explained_variance_

print("Eigenvectors:")
print('Energy, Kurtosis, Crest factor, Peak, Mean')
print(np.round(eigenvectors,2))

print("Eigenvalues:", np.round(eigenvalues,2))

print("Cluster labels:", labels_pca)
print(f"highest PC1: {np.argmax(centers_pca[:,0])}")
print(f'highest PC2: {np.argmax(centers_pca[:,1])}')
print(f'highest PC3: {np.argmax(centers_pca[:,2])}')

### 3d plot

3d plots of the results with axes energy, kurtosis, crest factor

In [ ]:
fig, ax = plt.subplots(2, figsize=(20, 20), subplot_kw={'projection': '3d'})

titles = ['Original Data', 'PCA Data']

# plot subplots for PCA data and non-PCA data
for i, ax in enumerate(ax):
    cmap = plt.cm.get_cmap('Set1')
    if i == 0:
        data_scatter = data_train
        centers_scatter = centers
        labels_scatter = labels
    else:
        data_scatter = data_train
        centers_scatter = centers_pca
        labels_scatter = labels_pca

    ax.scatter(
        [point[0] for point in data_scatter],
        [point[1] for point in data_scatter],
        [point[2] for point in data_scatter],
        c=[cmap(label) for label in labels_scatter])
    ax.scatter(
        centers_scatter[:, 0],
        centers_scatter[:, 1],
        centers_scatter[:, 2],
        c='red', marker='x', s=100)
    
    ax.set_xlabel('Energy')
    ax.set_ylabel('Kurtosis')
    ax.set_zlabel('Crest factor')
    ax.set_title(titles[i])

    # Annotate cluster centers with their positions
    for i, center in enumerate(centers_scatter):
        ax.text(center[0], center[1], center[2], f'Center {i}', color='black', fontsize=20)

plt.show()

### 2d plot

plots with energy vs crest factor comparing PCA and no PCA results

In [ ]:
# 3d plot
fig, ax = plt.subplots(2,figsize=(10,10))

# plot with colours for each cluster
cmap = plt.cm.get_cmap('Set1')
x = 0  # 0 -> energy, 1 -> kurtosis, 3 -> crest
y = 1  # ""
data_scatter = data_train
labels_scatter = labels
ax[0].scatter(
    [point[x] for point in data_scatter],
    [point[y] for point in data_scatter],
    c=[cmap(label) for label in labels_scatter]
)
labels_scatter = labels_pca
ax[1].scatter(
    [point[x] for point in data_scatter],
    [point[y] for point in data_scatter],
    c=[cmap(label) for label in labels_scatter]
)
# plot centres
centers_scatter = centers
ax[0].scatter(centers_scatter[:, x], 
           centers_scatter[:, y], c='red', marker='x', s=100)
centers_scatter = centers_pca
ax[1].scatter(centers_scatter[:, x], 
           centers_scatter[:, y], c='red', marker='x', s=100)
for i in range(2):
    ax[i].set_xlabel('Energy')
    ax[i].set_ylabel('Crest factor')
ax[0].set_title('Groupings without PCA')
ax[1].set_title('Groupings with PCA')
plt.show()

### Results

Plotting the reconstructed waves from clustering with their assigned group

In [ ]:
plot_reconstructions(rec, group=labels)

# Appendix

### kurtosis and crest factor calculations
testing kurtosis and crest factor calculations on wavelet packet coefficients and reconstructed waves

In [ ]:
# Calculate frequenciens for plotting
cent_freqs = [
    round(1 / 2 * sum(node.frequency_range), 1)
    for node in tr_obj.get_level(order="freq")
]
cent_freqs = np.array(cent_freqs)
sparse_cent_freqs = [
    cent_freq if id % 1 == 0 else "" for id, cent_freq in enumerate(cent_freqs)
]

# Gather data
coef_kurtosis = [node.metric("kurtosis") for node, *_ in rec]
kurtosis = [sc.kurtosis(i) for _, i, *_ in rec]

# plot bar chart
fig, ax = plt.subplots()
width = 0.4

fig.set_size_inches(18, 16)

scatter = ax.scatter(coef_kurtosis, kurtosis, c=cent_freqs)
ax.set_title("Kurtosis of node coefficients vs recovered signal", fontsize=25)
ax.set_xlabel("Kurtosis of transform coefficients", fontsize=20)
ax.set_ylabel("Kurtosis of recovered signal", fontsize=20)

fig.colorbar(scatter, ax=ax, label="Frequency [Hz]")

fig.tight_layout()
plt.show()

In [ ]:
# gather data
coef_spectral_entropy = [node.metric("spectral_entropy") for node, *_ in rec]
rec_spectral_entropy = [stats.spectral_entropy(i) for _, i, *_ in rec]

# plot bar chart
fig, ax = plt.subplots()
width = 0.4

fig.set_size_inches(18, 16)

scatter = ax.scatter(coef_spectral_entropy, rec_spectral_entropy, c=cent_freqs)
ax.set_title("Spectral entropy of node coefficients vs recovered signal", fontsize=25)
ax.set_xlabel("Spectral entropy of transform coefficients", fontsize=20)
ax.set_ylabel("Spectral entropy of recovered signal", fontsize=20)

fig.colorbar(scatter, ax=ax, label="Frequency [Hz]")

fig.tight_layout()
plt.show()

In [ ]:
# gather data
coef_crest = [node.metric("crest_factor") for node, *_ in rec]
crest = [stats.crest_factor(i) for _, i, *_ in rec]

# plot bar chart
fig, ax = plt.subplots()
width = 0.4

fig.set_size_inches(18, 16)

scatter = ax.scatter(coef_crest, crest, c=cent_freqs)
ax.set_title("Crest factor of node coefficients vs recovered signal", fontsize=25)
ax.set_xlabel("Crest factor of transform coefficients [dB]", fontsize=20)
ax.set_ylabel("Crest factor of recovered signal [dB]", fontsize=20)

fig.colorbar(scatter, ax=ax, label="Frequency [Hz]")

fig.tight_layout()
plt.show()

In [ ]:
# gather data
coef_entropy = [node.metric("entropy") for node, *_ in rec]
entropy = [stats.entropy(i) for _, i, *_ in rec]

# plot bar chart
fig, ax = plt.subplots()
width = 0.4

fig.set_size_inches(18, 16)

scatter = ax.scatter(coef_entropy, entropy, c=cent_freqs)
ax.set_title("Entropy of node coefficients vs recovered signal", fontsize=25)
ax.set_xlabel("Entropy of transform coefficients [dB]", fontsize=20)
ax.set_ylabel("Entropy of recovered signal [dB]", fontsize=20)

fig.colorbar(scatter, ax=ax, label="Frequency [Hz]")

fig.tight_layout()
plt.show()